<a href="https://colab.research.google.com/github/Ryan56-56/onix/blob/main/onix(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


In [8]:
df = pd.read_csv("/content/drive/MyDrive/data.csv", sep=';')

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip().str.replace('  ', ' ')

# 37 input features (same order as your HTML)
FEATURES = [
    'Marital status','Application mode','Application order','Course',
    'Daytime/evening attendance','Previous qualification',
    'Previous qualification (grade)','Nacionality',"Mother's qualification",
    "Father's qualification","Mother's occupation","Father's occupation",
    'Admission grade','Displaced','Educational special needs','Debtor',
    'Tuition fees up to date','Gender','Scholarship holder',
    'Age at enrollment','International',
    'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)','Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)','Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)','Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)','Curricular units 2nd sem (without evaluations)',
    'Unemployment rate','Inflation rate','GDP'
]

# Target (dropout prediction)
TARGET = "Target"   # <-- change to your actual column name

In [10]:
X = df[FEATURES].values.astype(np.float32)
# Convert 'Target' column from strings to numerical values (0 and 1)
y = df[TARGET].map({'Graduate': 0, 'Dropout': 1}).values.astype(np.float32).reshape(-1, 1)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [12]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled  = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)


In [13]:
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32)
)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)


In [16]:
class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(36, 64) # Changed from 37 to 36
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)

model = StudentNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [17]:
for epoch in range(200):
    for xb, yb in train_dl:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")


Epoch 0 Loss nan
Epoch 50 Loss nan
Epoch 100 Loss nan
Epoch 150 Loss nan


In [21]:
import torch

!pip install onnxscript

dummy = torch.randn(1, 36)

torch.onnx.export(
    model,
    dummy,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    export_params=True,
    do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)

print("Saved model.onnx")

/tmp/ipykernel_1775/2385163768.py:7: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/tmp/ipykernel_1775/2385163768.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0418 02:50:23.318000 1775 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >

[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved model.onnx
